# Week 08 — Auction simulation and bid shading

**Goal.** Build a first-price auction, bid your CVR model into it, and learn the shading policy that maximises surplus.

**Deliverable.** An auction simulator, a shading model, and a surplus-vs-aggressiveness curve.

**Rough shape of the week.** 2h reading (bid shading) · 6h building · 1h write-up.

---
### Ground rules (they apply every week)

1. **Beat a dumb baseline or it didn't happen.** Logistic regression or the global mean.
   Log the baseline in the same table as the fancy model.
2. **Split by time, never at random.** `split.time_split` — and call
   `split.check_no_leakage` so the assertion, not your memory, enforces it.
3. **Log every run** with `registry.log_result(...)`, including the ones that lost.
   The losing runs are what make the write-up honest.
4. **Write the finding down** in this week's `README.md` while it is fresh.

### Reading

PDFs are in `papers/` next to this notebook — see `papers/README.md`.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=FutureWarning)

%load_ext autoreload
%autoreload 2

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from adslab import data, metrics, plots, split, registry, encoders, calibration

plots.use_style()
pd.set_option("display.width", 140, "display.max_columns", 60)
print("harness ready")

## Why shading exists

In a second-price auction, bidding your true value is optimal — you pay the runner-up's
bid. First-price auctions, which the industry moved to, have no such guarantee: bid your
value and your surplus is exactly zero every time you win. So you shade: bid $b < v$,
trading win rate for margin.

The optimisation is
$$\max_b \;(v - b)\cdot \Pr(\text{win}\mid b)$$
and everything hinges on estimating $\Pr(\text{win}\mid b)$ — the win-rate curve — from
censored data. You only observe the winning price when you win. That censoring is the
real problem of the week.

In [ ]:
df = data.add_attribution_derived(data.load_attribution())
cost = df.cost[df.cost > 0]
print(f"observed price paid: p10={cost.quantile(.1):.5f} p50={cost.median():.5f} "
      f"p90={cost.quantile(.9):.5f}")

fig, ax = plt.subplots()
ax.hist(np.log10(cost.sample(200_000, random_state=0)), bins=60)
ax.set_xlabel("log10(price paid)"); ax.set_ylabel("count")
ax.set_title("Win-price distribution (Criteo, observed wins only)")
print(plots.save(fig, 8, "win_price_distribution"))

## 1. The simulator

Keep it honest and simple: N competitors per auction, each drawing a bid from a
distribution you control. Calibrate that distribution so the simulated win prices
resemble the Criteo `cost` histogram above — then you are testing your policy against
something with the right shape.

The design decision that matters: **your simulator knows the true competing bids, and
your bidder must not.** Enforce it in the interface, or you will accidentally write an
oracle and be delighted by your own results.

In [ ]:
class FirstPriceAuction:
    def __init__(self, n_competitors=5, price_dist="lognormal", seed=0):
        # TODO
        raise NotImplementedError

    def run(self, our_bid):
        """Return (won, price_paid, highest_competing_bid_IF_WE_WON_else_None)."""
        raise NotImplementedError

## 2. Value-based bidding, no shading

`bid = p_conversion * value_per_conversion * margin`, using your Week 1 model. Run it
through the simulator and record win rate, spend, conversions, and surplus. This is the
baseline every shading policy has to beat.

In [ ]:
# TODO

## 3. Learn the win-rate curve from censored data

The core exercise. You observe the price only on wins, so a naive fit to observed prices
is biased upward — it is a survival problem, not a regression.

Two approaches, do at least one properly:
- **Parametric**: assume the highest competing bid is lognormal and fit by maximum
  likelihood with right-censoring on losses.
- **Non-parametric**: Kaplan–Meier on the censored bid landscape.

Then verify against the simulator's known truth. That verification is the whole reason
to have built a simulator instead of only using real data.

In [ ]:
# TODO

## 4. Optimise the shade

With $\hat{\Pr}(\text{win}\mid b)$ in hand, maximise $(v-b)\Pr(\text{win}\mid b)$ per
impression. Compare against fixed-multiplier shading (bid $0.7v$, $0.8v$, ...).

The plot: surplus vs. shading aggressiveness, with the learned policy marked. If your
learned policy does not beat the best fixed multiplier, say so — it is a real and common
result, and it means the win-rate curve is not varying enough per impression to be worth
modelling.

In [ ]:
# fig, ax = plt.subplots()
# ... surplus vs multiplier, learned policy as a horizontal line
# print(plots.save(fig, 8, "surplus_vs_shading"))

## 5. Feed it a miscalibrated model

The link back to Week 3, and the best experiment in this notebook. Take your Week 3
uncalibrated model — the one with a `calibration_ratio` of 1.3 — and bid with it.
Quantify the overspend in currency.

This is the moment the calibration work stops being an academic metric and becomes money.

In [ ]:
# TODO: rerun the auction with calibrated vs uncalibrated predictions, compare spend & CPA

---
## Log the results

Every model you tried, including the baseline and including the failures. `notes` is the
one sentence you would say out loud about the run — future-you assembles the write-up
from these, so write it now while you still remember why the run mattered.

In [ ]:
# registry.log_result(
#     week=8,
#     model="lightgbm_hashed_2^18",
#     metrics=metrics.evaluate(y_test, p_test),
#     dataset="attribution",
#     params=dict(n_bits=18, num_leaves=63, lr=0.05),
#     notes="beats LR by 0.011 AUC; most of the gain is from cat3 x cat7 interactions",
# )

print(registry.to_markdown(week=8))

---
## Write it up

Open `README.md` in this folder and fill in the three sections. Keep it to a page.

- **What I built** — one paragraph, no code.
- **What the numbers say** — paste the table above; say which comparison is the honest one.
- **What surprised me** — the part worth reading. If nothing surprised you, you probably
  did not stress the model hard enough.

Then commit:

```bash
git add week08_* results/
git commit -m "week 08: <the finding, not the task>"
```